In [2]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

# agiso@dtu.dk
using JuMP, HiGHS


   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`


In [11]:
##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

nodes = 6

arches = zeros(Int, nodes, nodes)
cost = zeros(Int, nodes, nodes)


arches[1,2], cost[1,2] = 1, 6
arches[1,5], cost[1,5] = 1, 2

arches[2,3], cost[2,3] = 1, 5
arches[2,6], cost[2,6] = 1, 2

arches[3,4], cost[3,4] = 1, 5

arches[5,2], cost[5,2] = 1, 5
arches[5,3], cost[5,3] = 1, 2
arches[5,6], cost[5,6] = 1, 5

arches[6,3], cost[6,3] = 1, 6
arches[6,4], cost[6,4] = 1, 10

source = 1
sink = 4

########## ---------- Variables ---------- ##########
@variable(model, x[1:nodes, 1:nodes], Bin)

########## ---------- Objective ---------- ##########
@objective(model, Min, sum(x[i, j] * cost[i,j] for i in 1:nodes, j in 1:nodes) )

########## ---------- Constrait ---------- ##########
# Flow constraint for intermediate nodes
# from == to
@constraint(model, [k in 1:nodes; k != source && k != sink],
    sum(x[k,j] for j in 1:nodes) - sum(x[j,k] for j in 1:nodes) == 0
)

# Flow for source
# from - to = 1
@constraint(model,
    sum(x[source,j] for j in 1:nodes) - sum(x[j,source] for j in 1:nodes) == 1
)

# Flow for sink
# from - to = -1
@constraint(model,
    sum(x[sink,j] for j in 1:nodes) - sum(x[j,sink] for j in 1:nodes) == -1
)

# If it's not an edge, don't use it
@constraint(model, [i in 1:nodes, j in 1:nodes],
    x[i,j] <= arches[i,j]
)

########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))

for i in 1:nodes
    println(value.(x[i, :]))
end

Optimal solution:
9.0
[0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [ ]:
nodes = 10
arches = zeros(Int, nodes, nodes)
cost   = zeros(Int, nodes, nodes)

# From node 4
arches[4,3],  cost[4,3]   = 1, 8
arches[4,5],  cost[4,5]   = 1, 5

# From node 8  (outer arc into 4)
arches[8,4],  cost[8,4]   = 1, 3
arches[8,10], cost[8,10]  = 1, 5

# From node 3
arches[3,6],  cost[3,6]   = 1, 9
arches[3,10], cost[3,10]  = 1, 9
arches[3,1],  cost[3,1]   = 1, 3
arches[3,8],  cost[3,8]   = 1, 8

# From node 6
arches[6,9],  cost[6,9]   = 1, 3

# From node 5
arches[5,9],  cost[5,9]   = 1, 7

# From node 9
arches[9,10], cost[9,10]  = 1, 6
arches[9,7],  cost[9,7]   = 1, 1
arches[9,1],  cost[9,1]   = 1, 7

# From node 10
arches[10,7], cost[10,7]  = 1, 7

# From node 7
arches[7,2],  cost[7,2]   = 1, 1
arches[7,1],  cost[7,1]   = 1, 8

# From node 1
arches[1,5],  cost[1,5]   = 1, 5
arches[1,4],  cost[1,4]   = 1, 4


demands = [
    30 30 50 -250 20 30 30 40 0 20
]

########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, x[1:nodes, 1:nodes] >= 0)

########## ---------- Objective ---------- ##########
@objective(model, Min, sum(x[i, j] * cost[i,j] for i in 1:nodes, j in 1:nodes) )

########## ---------- contraint ---------- ##########
# Sum of outgoing - sum of ingoing = - demand (ie. supply)
@constraint(model, [i in 1:nodes],
    sum(x[j,i] for j in 1:nodes) - sum(x[i,j] for j in 1:nodes) == -1 * demands[i]
)

########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))
for i in 1:nodes
    println(value.(x[i, :]))
end

LoadError: LoadError: At In[18]:56: `@constraint(model, [i in 1:nodes], sum((x[(j, i)] for (j, i) = A if i == i)))`: Unsupported constraint expression: we don't know how to parse constraints containing the operator sum.

If you are writing a JuMP extension, implement `parse_constraint_call(::Function, ::Bool, ::Val{sum}, args...)
in expression starting at In[18]:56

# Multi commodity flow

In [ ]:
using JuMP, HiGHS

# ============================================================
# Multi-Commodity Flow (MCF) LP for container shipping
# ============================================================
# Ports (nodes) are indexed 1..m:
#  1 ROT (Rotterdam)     2 HAM (Hamburg)      3 FEL (Felixstowe)
#  4 ALG (Algeciras)     5 DXB (Jebel Ali)    6 SIN (Singapore)
#  7 SHA (Shanghai)      8 BUS (Busan)        9 LAX (Los Angeles)
# 10 PAN (Panama)       11 NYC (New York)
# ============================================================

ports = [
    "ROT", "HAM", "FEL", "ALG", "DXB", "SIN", "SHA", "BUS", "LAX", "PAN", "NYC"
]
m = length(ports)

# ------------------------------------------------------------
# arcs: defined by ship rotation legs (directed)
# ------------------------------------------------------------

# Define arcs by rotation (as directed legs)
arcs = [(1,2), (2,3), (3,4), (4,1),                  # ROT->HAM->FEL->ALG->ROT
        (4,5), (5,6), (6,4),                         # ALG->DXB->SIN->ALG
        (6,7), (7,8), (8,6),                         # SIN->SHA->BUS->SIN
        (7,9), (9,10), (10,11), (11,7),              # SHA->LAX->PAN->NYC->SHA  
        (1,11), (11,10), (10,1),                     # ROT->NYC->PAN->ROT
        (2,5), (5,7), (7,6), (6,2)]                  # HAM->DXB->SHA->SIN->HAM

# Capacity per directed arc a 
cap = Dict{Tuple{Int,Int}, Float64}()
# ROT->HAM->FEL->ALG->ROT
cap[(1,2)] = 800.0
cap[(2,3)] = 800.0
cap[(3,4)] = 800.0
cap[(4,1)] = 800.0

# ALG->DXB->SIN->ALG
cap[(4,5)] = 1100.0
cap[(5,6)] = 1100.0
cap[(6,4)] = 1100.0

# SIN->SHA->BUS->SIN
cap[(6,7)] = 1400.0
cap[(7,8)] = 1400.0
cap[(8,6)] = 1400.0

# SHA->LAX->PAN->NYC->SHA
cap[(7,9)] = 1000.0
cap[(9,10)] = 1000.0
cap[(10,11)] = 1000.0
cap[(11,7)] = 1000.0

# ROT->NYC->PAN->ROT
cap[(1,11)] = 900.0
cap[(11,10)] = 900.0
cap[(10,1)] = 900.0

# HAM->DXB->SHA->SIN->HAM
cap[(2,5)] = 1200.0
cap[(5,7)] = 1200.0
cap[(7,6)] = 1200.0
cap[(6,2)] = 1200.0

# cost matrix:
c_matrix =
[0	4.5	3	22	65	105	195	200	160	88	62
4.5	0	5	24	66	107	197	202	162	90	64
3	5	0	21	64	104	194	199	158	86	61
22	24	21	0	52	85	165	170	150	82	65
65	66	64	52	0	58	102	108	135	150	110
105	107	104	85	58	0	38	46	141	167	155
195	197	194	165	102	38	0	9	104	125	190
200	202	199	170	108	46	9	0	98	120	185
160	162	158	150	135	141	104	98	0	48	63
88	90	86	82	150	167	125	120	48	0	35
62	64	61	65	110	155	190	185	63	35	0]

# ------------------------------------------------------------
# Demands (commodities): (origin, destination, volume TEU/week)
# ------------------------------------------------------------
demands = [
    (2, 7, 480.0),  # k1: Hamburg -> Shanghai
    (1, 6, 400.0),  # k2: Rotterdam -> Singapore
    (3, 5,  320.0),  # k3: Felixstowe -> Jebel Ali
    (7, 11, 560.0), # k4: Shanghai -> New York
    (6, 10,  360.0), # k5: Singapore -> Panama
    (1, 11, 440.0), # k6: Rotterdam -> New York
    (2, 9, 520.0),   # k7: Hamburg -> Los Angeles
    (8, 4, 300.0)   # k8: Busan -> Algeciras
]
K = length(demands)

# ------------------------------------------------------------
# Build LP model
# ------------------------------------------------------------
model = Model(HiGHS.Optimizer)
set_silent(model)

########## ---------- Variables ---------- ##########
@variable(model, x[i,j,k] >= 1)

########## ---------- Objectives ---------- ##########
@objective(mdoel, Min, sum(c_matrix[i,j] * x[i,j,k] for i in 1:m, j in 1:m, k in 1:K ))

########## ---------- Constraint ---------- ##########
for k in 1:K
    src, dest, qty = demands[k]
    @constraint(model, 
        sum([x[i,,k]])
    )
end
